# 26장 만든 모델 보여주기 — Gradio

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `app.py` 와 `test_app.py` 를 치고, `test_app.py` 먼저 → `app.py` 실행 → 브라우저에서 그리기 | [ch26_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `predict` 각 줄에 텐서 shape과 "왜 반전하는가"를 주석으로 | [ch26_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `predict` 복원. 그다음 확인 문제 5(조문 분류기 화면) | [ch26_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "26"))                      # 이 장의 코드 폴더로 이동
if "google.colab" in sys.modules:
    !pip install -q gradio
if not os.path.exists("../13/results/lr01/model_best.pt"):      # 13장에서 학습한 모델이 없으면 만든다
    !cd ../13 && python train.py --run lr01
print("준비 끝. 작업 폴더: code/26")

준비 끝. 작업 폴더: code/26


## `app.py` — 26장 만든 모델 보여주기: 13장에서 학습한 MNIST 분류기를 그림판이 달린 웹 화면으로

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 26장 만든 모델 보여주기: 13장에서 학습한 MNIST 분류기를 그림판이 달린 웹 화면으로
# 준비: pip install gradio      실행: python app.py  → 브라우저에서 http://127.0.0.1:7860
import os
import sys
import json
import numpy as np
import torch
import gradio as gr
from PIL import Image

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "13"))
from train import MLP                                     # 13장의 모델 정의를 그대로

RUN_DIR = os.path.join(os.path.dirname(__file__), "..", "13", "results", "lr01")


def load_model():
    with open(os.path.join(RUN_DIR, "config.json"), encoding="utf-8") as f:
        cfg = json.load(f)
    m = MLP(hidden=cfg["hidden"])
    m.load_state_dict(torch.load(os.path.join(RUN_DIR, "model_best.pt"), map_location="cpu"))
    m.eval()
    return m


model = load_model()


def predict(image):
    """그림판 입력(딕셔너리 또는 배열) → 28x28 흑백 → 확률 10개"""
    if image is None:
        return {}
    if isinstance(image, dict):                            # Sketchpad 는 {"composite": 배열} 형태
        image = image["composite"]
    img = Image.fromarray(np.asarray(image)).convert("L").resize((28, 28))
    x = torch.tensor(np.asarray(img), dtype=torch.float32) / 255.0
    if x.mean() > 0.5:                                     # 흰 바탕에 검은 글씨면 반전 (MNIST 는 검은 바탕)
        x = 1.0 - x
    with torch.no_grad():
        probs = torch.softmax(model(x[None, None]), dim=1)[0]
    return {str(i): float(p) for i, p in enumerate(probs)}


demo = gr.Interface(
    fn=predict,
    inputs=gr.Sketchpad(label="숫자를 그려 보세요", image_mode="L"),
    outputs=gr.Label(num_top_classes=3, label="예측"),
    title="손글씨 숫자 분류기 (12·13장 MLP)",
    description="13장에서 학습한 모델(시험 정확도 94.7%)입니다. 그림판에 0~9를 그리면 확률을 보여 줍니다.",
)

if __name__ == "__main__":
    demo.launch()                                          # share=True 를 주면 1주일짜리 공개 링크가 생긴다
```

## `test_app.py` — 26장 검증: 브라우저 없이 app.py 의 predict 함수와 서버 기동을 확인한다 (6장의 '시험한다').

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 26장 검증: 브라우저 없이 app.py 의 predict 함수와 서버 기동을 확인한다 (6장의 '시험한다').
import os
import sys
import numpy as np
import torch
from torchvision import datasets, transforms

sys.path.insert(0, os.path.dirname(__file__))
import app

# 1) predict 가 시험 이미지에 대해 13장 infer.py 와 같은 답을 내는가
test_set = datasets.MNIST(os.environ.get("JUCOS_DATA", "./data"), train=False, download=True)
for i in (0, 1, 2, 8):
    img, label = test_set[i]                               # PIL, 검은 바탕 흰 글씨
    arr = 255 - np.asarray(img)                            # 그림판처럼 흰 바탕 검은 글씨로 뒤집어 넣는다
    probs = app.predict(arr)
    pred = max(probs, key=probs.get)
    print(f"시험 이미지 {i}: 정답 {label}  예측 {pred}  확신 {probs[pred]:.3f}")

# 2) 서버가 뜨고 HTTP 로 응답하는가
app.demo.launch(prevent_thread_lock=True, quiet=True)
import urllib.request
with urllib.request.urlopen("http://127.0.0.1:7860/", timeout=10) as r:
    html = r.read().decode("utf-8", "ignore")
print("서버 응답:", r.status, "| 제목 포함:", "손글씨" in html or "gradio" in html.lower())
app.demo.close()
print("서버 종료")
```

In [ ]:
!python test_app.py

시험 이미지 0: 정답 7  예측 7  확신 0.996
시험 이미지 1: 정답 2  예측 2  확신 0.984
시험 이미지 2: 정답 1  예측 1  확신 0.979
시험 이미지 8: 정답 5  예측 6  확신 0.976
서버 응답: 200 | 제목 포함: True
Closing server running on port: 7860
서버 종료


## 화면 띄우기

Colab 에서는 아래 셀을 실행하면 셀 아래에 그림판이 나타나고, 임시 공개 주소도 함께 만들어집니다. 숫자를 그려 예측 확률을 보세요. 끝나면 그 아래 셀로 서버를 닫습니다.

In [ ]:
import app
app.demo.launch()

In [ ]:
app.demo.close()

## 다음 단계

- 2단계 「주석 달기」: [ch26_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch26_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch26_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.